In [ ]:
!pip install -q \
    numpy<2.0.0 \
    transformers==4.40.0 \
    datasets==2.19.0 \
    torch==2.2.1 \
    torchaudio==2.2.1 \
    librosa==0.10.1 \
    soundfile==0.12.1 \
    shap==0.45.0 \
    fastapi==0.111.0 \
    uvicorn==0.29.0 \
    python-multipart==0.0.9 \
    pyngrok==7.1.6 \
    audiomentations==0.34.1 \
    nest-asyncio==1.6.0 \
    accelerate==0.30.1 \
    evaluate==0.4.2 \
    scikit-learn==1.4.2

print('✅ All packages installed')

/bin/bash: line 1: 2.0.0: No such file or directory


ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-132' coro=<Server.serve() done, defined at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:67> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_27358/2946195640.py", line 119, in <cell line: 0>
    uvicorn.run(app, host="127.0.0.1", port=8000)
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 575, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 65, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/us

✅ All packages installed


In [ ]:
!pip install "numpy<2.0.0"

In [ ]:
import torch
import os

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found — go to Runtime > Change runtime type > T4 GPU')

CONFIG = {
    'bert_model':    'bert-base-uncased',
    'wav2vec_model': 'facebook/wav2vec2-base',
    'dataset':       'declare-lab/meld',
    'max_text_len':  128,
    'sample_rate':   16000,
    'max_audio_sec': 6,
    'bert_epochs':      4,
    'wav2vec_epochs':   4,
    'fusion_epochs':    6,
    'batch_size':       16,
    'lr_bert':          2e-5,
    'lr_wav2vec':       1e-4,
    'lr_fusion':        3e-4,
    'emotion_to_sentiment': {0:'neutral', 1:'mixed', 2:'negative',
                              3:'negative', 4:'positive', 5:'negative', 6:'negative'},
    'sentiment_labels': ['positive', 'negative', 'neutral', 'mixed'],
    'num_labels': 4,
    'conflict_threshold': 0.38,
    'save_dir': '/content/drive/MyDrive/SonicSense',
    'local_dir': '/content/sonicsense_models',
}

os.makedirs(CONFIG['local_dir'], exist_ok=True)
print('\n✅ Config ready')
print(f"   Save directory: {CONFIG['local_dir']}")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB

✅ Config ready
   Save directory: /content/sonicsense_models


In [ ]:
import torch
from transformers import BertTokenizerFast, Wav2Vec2Processor, BertForSequenceClassification, Wav2Vec2ForSequenceClassification
import torch.nn as nn

# 1. Reload text model
bert_tokenizer = BertTokenizerFast.from_pretrained(f"{CONFIG['local_dir']}/bert_best")
bert_model = BertForSequenceClassification.from_pretrained(f"{CONFIG['local_dir']}/bert_best").to(DEVICE)

# 2. Reload audio model
wav2vec_proc = Wav2Vec2Processor.from_pretrained(f"{CONFIG['local_dir']}/wav2vec_best")
wav2vec_model = Wav2Vec2ForSequenceClassification.from_pretrained(f"{CONFIG['local_dir']}/wav2vec_best").to(DEVICE)

# 3. Rebuild and reload fusion model
class CrossAttentionFusion(nn.Module):
    def __init__(self, num_labels=4, hidden=128, nheads=2, dropout=0.2):
        super().__init__()
        self.text_proj  = nn.Linear(num_labels, hidden)
        self.audio_proj = nn.Linear(num_labels, hidden)
        self.cross_attn = nn.MultiheadAttention(embed_dim=hidden, num_heads=nheads, dropout=dropout, batch_first=True)
        self.norm    = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, num_labels))

    def forward(self, text_logits, audio_logits):
        t = self.text_proj(text_logits).unsqueeze(1)
        a = self.audio_proj(audio_logits).unsqueeze(1)
        ta, attn_w = self.cross_attn(query=t, key=a, value=a)
        at, _      = self.cross_attn(query=a, key=t, value=t)
        ta = self.norm(ta.squeeze(1) + t.squeeze(1))
        at = self.norm(at.squeeze(1) + a.squeeze(1))
        fused  = torch.cat([ta, at], dim=-1)
        fused  = self.dropout(fused)
        logits = self.head(fused)
        attn_w = attn_w.squeeze(1)
        text_weight  = attn_w.mean(dim=-1).squeeze(-1)
        audio_weight = 1.0 - text_weight
        return logits, text_weight, audio_weight

fusion_model = CrossAttentionFusion(num_labels=CONFIG['num_labels']).to(DEVICE)
fusion_model.load_state_dict(torch.load(f"{CONFIG['local_dir']}/fusion_best.pt"))

# Restore required variables for Cell 9
REAL_ACCURACY = {'text_only': 0.0, 'audio_only': 0.0, 'fusion': 0.0}
MAX_AUDIO_FRAMES = CONFIG['sample_rate'] * CONFIG['max_audio_sec']
SENT2IDX = {s: i for i, s in enumerate(CONFIG['sentiment_labels'])}
IDX2SENT = {i: s for s, i in SENT2IDX.items()}

print("✅ All models successfully reloaded from disk!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/214 [00:00<?, ?it/s]

✅ All models successfully reloaded from disk!


In [ ]:
!pip install audiomentations==0.34.1

In [ ]:
import numpy as np
import librosa
from datasets import load_dataset
from transformers import BertTokenizerFast, Wav2Vec2Processor
from torch.utils.data import Dataset, DataLoader

print('📥 Connecting to Pre-processed MELD dataset in STREAMING mode...')
raw = load_dataset('Vano04/MELD-Preprocessed', streaming=True)

splits = list(raw.keys())
print(f"   Found folders: {splits}")
print('✂️ Pulling lightweight subsets directly from the stream...')

train_stream = list(raw['train'].take(1700))
train_data = train_stream[:1500]
val_data   = train_stream[1500:1700]
test_data  = list(raw['test'].take(200))

print(f'   Train size: {len(train_data)}')

bert_tokenizer   = BertTokenizerFast.from_pretrained(CONFIG['bert_model'])
wav2vec_proc     = Wav2Vec2Processor.from_pretrained(CONFIG['wav2vec_model'])
SR               = CONFIG['sample_rate']
MAX_AUDIO_FRAMES = SR * CONFIG['max_audio_sec']

SENT2IDX = {s: i for i, s in enumerate(CONFIG['sentiment_labels'])}
IDX2SENT = {i: s for s, i in SENT2IDX.items()}

def get_sentiment_label(emotion_str):
    if isinstance(emotion_str, str):
        emotion_str = emotion_str.lower().strip()
        if emotion_str in ['joy', 'happy', 'positive']: return SENT2IDX['positive']
        if emotion_str in ['anger', 'sadness', 'disgust', 'fear', 'negative']: return SENT2IDX['negative']
        if emotion_str in ['surprise', 'mixed']: return SENT2IDX['mixed']
        return SENT2IDX['neutral']
    else:
        return SENT2IDX[CONFIG['emotion_to_sentiment'].get(emotion_str, 0)]

class MELDDataset(Dataset):
    def __init__(self, data_list, augment=False):
        self.data    = data_list
        self.augment = augment

        if augment:
            from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
            self.aug = Compose([
                AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.4),
                TimeStretch(min_rate=0.9, max_rate=1.1, p=0.3),
                PitchShift(min_semitones=-2, max_semitones=2, p=0.3),
            ])
        else:
            self.aug = None

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        text_val = item.get('Utterance', item.get('text', item.get('sentence', '')))
        emotion_val = item.get('Emotion', item.get('label', item.get('emotion', 'neutral')))
        audio_dict = item.get('audio', item.get('audio_values', {}))

        text_enc = bert_tokenizer(
            text_val,
            max_length=CONFIG['max_text_len'],
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        try:
            audio_array = audio_dict.get('array', np.zeros(SR, dtype=np.float32)).astype(np.float32)
            orig_sr     = audio_dict.get('sampling_rate', SR)

            if orig_sr != SR and len(audio_array) > 0:
                audio_array = librosa.resample(audio_array, orig_sr=orig_sr, target_sr=SR)
        except Exception:
            audio_array = np.zeros(SR, dtype=np.float32)

        if len(audio_array) < MAX_AUDIO_FRAMES:
            audio_array = np.pad(audio_array, (0, MAX_AUDIO_FRAMES - len(audio_array)))
        else:
            audio_array = audio_array[:MAX_AUDIO_FRAMES]

        if self.aug is not None:
            audio_array = self.aug(audio_array, sample_rate=SR)

        audio_enc = wav2vec_proc(
            audio_array,
            sampling_rate=SR,
            return_tensors='pt',
            padding=True
        )

        label = get_sentiment_label(emotion_val)

        return {
            'input_ids':      text_enc['input_ids'].squeeze(0),
            'attention_mask': text_enc['attention_mask'].squeeze(0),
            'audio_values':   audio_enc['input_values'].squeeze(0),
            'label':          torch.tensor(label, dtype=torch.long),
        }

train_ds  = MELDDataset(train_data, augment=True)
val_ds    = MELDDataset(val_data,   augment=False)
test_ds   = MELDDataset(test_data,  augment=False)

BS = CONFIG['batch_size']
train_dl = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BS, shuffle=False, num_workers=0, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BS, shuffle=False, num_workers=0, pin_memory=True)

print(f'\n✅ Datasets ready')
print(f'   Label map: {IDX2SENT}')

📥 Connecting to Pre-processed MELD dataset in STREAMING mode...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


   Found folders: ['train', 'test']
✂️ Pulling lightweight subsets directly from the stream...
   Train size: 1500

✅ Datasets ready
   Label map: {0: 'positive', 1: 'negative', 2: 'neutral', 3: 'mixed'}


In [ ]:
from transformers import BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score

print('🔵 Loading BERT...')
bert_model = BertForSequenceClassification.from_pretrained(
    CONFIG['bert_model'],
    num_labels=CONFIG['num_labels'],
    hidden_dropout_prob=0.1,
).to(DEVICE)

bert_optim = AdamW(bert_model.parameters(), lr=CONFIG['lr_bert'], weight_decay=0.01)
total_steps = len(train_dl) * CONFIG['bert_epochs']
bert_sched  = get_linear_schedule_with_warmup(
    bert_optim,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

def train_epoch(model, dl, optim, sched, model_name):
    model.train()
    total_loss, preds_all, labels_all = 0, [], []
    for step, batch in enumerate(dl):
        input_ids  = batch['input_ids'].to(DEVICE)
        attn_mask  = batch['attention_mask'].to(DEVICE)
        labels     = batch['label'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        loss    = outputs.loss

        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        sched.step()

        total_loss += loss.item()
        preds_all.extend(outputs.logits.argmax(-1).cpu().tolist())
        labels_all.extend(labels.cpu().tolist())

        if (step + 1) % 50 == 0:
            print(f'   [{model_name}] step {step+1}/{len(dl)}  loss={loss.item():.4f}')

    acc = accuracy_score(labels_all, preds_all)
    f1  = f1_score(labels_all, preds_all, average='weighted')
    return total_loss / len(dl), acc, f1

def eval_epoch(model, dl):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for batch in dl:
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)
            labels    = batch['label'].to(DEVICE)
            outputs   = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds_all.extend(outputs.logits.argmax(-1).cpu().tolist())
            labels_all.extend(labels.cpu().tolist())
    acc = accuracy_score(labels_all, preds_all)
    f1  = f1_score(labels_all, preds_all, average='weighted')
    return total_loss / len(dl), acc, f1

best_val_f1 = 0
print(f'\n🚀 Training BERT for {CONFIG["bert_epochs"]} epochs...')
for epoch in range(CONFIG['bert_epochs']):
    tr_loss, tr_acc, tr_f1 = train_epoch(bert_model, train_dl, bert_optim, bert_sched, 'BERT')
    vl_loss, vl_acc, vl_f1 = eval_epoch(bert_model, val_dl)
    print(f'  Epoch {epoch+1}/{CONFIG["bert_epochs"]}  '
          f'train_loss={tr_loss:.4f} train_acc={tr_acc:.3f} | '
          f'val_loss={vl_loss:.4f} val_acc={vl_acc:.3f} val_f1={vl_f1:.3f}')
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        bert_model.save_pretrained(f"{CONFIG['local_dir']}/bert_best")
        bert_tokenizer.save_pretrained(f"{CONFIG['local_dir']}/bert_best")
        print(f'  💾 Saved best BERT (val_f1={vl_f1:.3f})')

bert_model = BertForSequenceClassification.from_pretrained(
    f"{CONFIG['local_dir']}/bert_best").to(DEVICE)
print(f'\n✅ BERT training complete. Best val F1: {best_val_f1:.3f}')

🔵 Loading BERT...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🚀 Training BERT for 4 epochs...
   [BERT] step 50/94  loss=0.0432
  Epoch 1/4  train_loss=0.5050 train_acc=0.831 | val_loss=0.0018 val_acc=1.000 val_f1=1.000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  💾 Saved best BERT (val_f1=1.000)
   [BERT] step 50/94  loss=0.0015
  Epoch 2/4  train_loss=0.0015 train_acc=1.000 | val_loss=0.0008 val_acc=1.000 val_f1=1.000
   [BERT] step 50/94  loss=0.0009
  Epoch 3/4  train_loss=0.0009 train_acc=1.000 | val_loss=0.0005 val_acc=1.000 val_f1=1.000
   [BERT] step 50/94  loss=0.0007
  Epoch 4/4  train_loss=0.0007 train_acc=1.000 | val_loss=0.0005 val_acc=1.000 val_f1=1.000


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


✅ BERT training complete. Best val F1: 1.000


In [ ]:
from transformers import Wav2Vec2ForSequenceClassification
import torch.nn as nn

print('🎵 Loading Wav2Vec2...')

wav2vec_model = Wav2Vec2ForSequenceClassification.from_pretrained(
    CONFIG['wav2vec_model'],
    num_labels=CONFIG['num_labels'],
    hidden_dropout=0.1,         # ← Changed from hidden_dropout_prob
    attention_dropout=0.1,
    mask_time_prob=0.0,
).to(DEVICE)

wav2vec_model.freeze_feature_encoder()

wav2vec_optim = AdamW(
    filter(lambda p: p.requires_grad, wav2vec_model.parameters()),
    lr=CONFIG['lr_wav2vec'],
    weight_decay=0.01
)
total_steps_wv = len(train_dl) * CONFIG['wav2vec_epochs']
wav2vec_sched  = get_linear_schedule_with_warmup(
    wav2vec_optim,
    num_warmup_steps=int(0.06 * total_steps_wv),
    num_training_steps=total_steps_wv
)

def train_epoch_audio(model, dl, optim, sched):
    model.train()
    total_loss, preds_all, labels_all = 0, [], []
    for step, batch in enumerate(dl):
        audio  = batch['audio_values'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        outputs = model(input_values=audio, labels=labels)
        loss    = outputs.loss

        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        sched.step()

        total_loss += loss.item()
        preds_all.extend(outputs.logits.argmax(-1).cpu().tolist())
        labels_all.extend(labels.cpu().tolist())

        if (step + 1) % 50 == 0:
            print(f'   [Wav2Vec] step {step+1}/{len(dl)}  loss={loss.item():.4f}')

    acc = accuracy_score(labels_all, preds_all)
    f1  = f1_score(labels_all, preds_all, average='weighted')
    return total_loss / len(dl), acc, f1

def eval_epoch_audio(model, dl):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for batch in dl:
            audio   = batch['audio_values'].to(DEVICE)
            labels  = batch['label'].to(DEVICE)
            outputs = model(input_values=audio, labels=labels)
            total_loss += outputs.loss.item()
            preds_all.extend(outputs.logits.argmax(-1).cpu().tolist())
            labels_all.extend(labels.cpu().tolist())
    acc = accuracy_score(labels_all, preds_all)
    f1  = f1_score(labels_all, preds_all, average='weighted')
    return total_loss / len(dl), acc, f1

best_wv_f1 = 0
print(f'\n🚀 Training Wav2Vec2 for {CONFIG["wav2vec_epochs"]} epochs...')
for epoch in range(CONFIG['wav2vec_epochs']):
    tr_loss, tr_acc, tr_f1 = train_epoch_audio(wav2vec_model, train_dl, wav2vec_optim, wav2vec_sched)
    vl_loss, vl_acc, vl_f1 = eval_epoch_audio(wav2vec_model, val_dl)
    print(f'  Epoch {epoch+1}/{CONFIG["wav2vec_epochs"]}  '
          f'train_loss={tr_loss:.4f} train_acc={tr_acc:.3f} | '
          f'val_loss={vl_loss:.4f} val_acc={vl_acc:.3f} val_f1={vl_f1:.3f}')
    if vl_f1 > best_wv_f1:
        best_wv_f1 = vl_f1
        wav2vec_model.save_pretrained(f"{CONFIG['local_dir']}/wav2vec_best")
        wav2vec_proc.save_pretrained(f"{CONFIG['local_dir']}/wav2vec_best")
        print(f'  💾 Saved best Wav2Vec2 (val_f1={vl_f1:.3f})')

wav2vec_model = Wav2Vec2ForSequenceClassification.from_pretrained(
    f"{CONFIG['local_dir']}/wav2vec_best").to(DEVICE)
print(f'\n✅ Wav2Vec2 training complete. Best val F1: {best_wv_f1:.3f}')

🎵 Loading Wav2Vec2...


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
wav2vec2.masked_spec_embed   | UNEXPECTED | 
projector.weight             | MISSING    | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 
projector.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🚀 Training Wav2Vec2 for 4 epochs...
   [Wav2Vec] step 50/94  loss=0.0072
  Epoch 1/4  train_loss=0.1934 train_acc=0.968 | val_loss=0.0015 val_acc=1.000 val_f1=1.000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  💾 Saved best Wav2Vec2 (val_f1=1.000)
   [Wav2Vec] step 50/94  loss=0.0011
  Epoch 2/4  train_loss=0.0012 train_acc=1.000 | val_loss=0.0006 val_acc=1.000 val_f1=1.000
   [Wav2Vec] step 50/94  loss=0.0006
  Epoch 3/4  train_loss=0.0006 train_acc=1.000 | val_loss=0.0004 val_acc=1.000 val_f1=1.000
   [Wav2Vec] step 50/94  loss=0.0005
  Epoch 4/4  train_loss=0.0005 train_acc=1.000 | val_loss=0.0004 val_acc=1.000 val_f1=1.000


Loading weights:   0%|          | 0/214 [00:00<?, ?it/s]


✅ Wav2Vec2 training complete. Best val F1: 1.000


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CrossAttentionFusion(nn.Module):
    def __init__(self, num_labels=4, hidden=128, nheads=2, dropout=0.2):
        super().__init__()
        self.text_proj  = nn.Linear(num_labels, hidden)
        self.audio_proj = nn.Linear(num_labels, hidden)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden, num_heads=nheads,
            dropout=dropout, batch_first=True
        )

        self.norm    = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_labels)
        )

    def forward(self, text_logits, audio_logits):
        t = self.text_proj(text_logits).unsqueeze(1)
        a = self.audio_proj(audio_logits).unsqueeze(1)

        ta, attn_w = self.cross_attn(query=t, key=a, value=a)
        at, _      = self.cross_attn(query=a, key=t, value=t)

        ta = self.norm(ta.squeeze(1) + t.squeeze(1))
        at = self.norm(at.squeeze(1) + a.squeeze(1))

        fused  = torch.cat([ta, at], dim=-1)
        fused  = self.dropout(fused)
        logits = self.head(fused)

        attn_w = attn_w.squeeze(1)
        text_weight  = attn_w.mean(dim=-1).squeeze(-1)
        audio_weight = 1.0 - text_weight

        return logits, text_weight, audio_weight


fusion_model = CrossAttentionFusion(
    num_labels=CONFIG['num_labels']
).to(DEVICE)

fusion_optim = AdamW(fusion_model.parameters(), lr=CONFIG['lr_fusion'], weight_decay=0.01)
ce_loss      = nn.CrossEntropyLoss()

def extract_logits(dl, bert_m, wav2vec_m):
    bert_m.eval(); wav2vec_m.eval()
    all_tl, all_al, all_lab = [], [], []
    with torch.no_grad():
        for batch in dl:
            tl = bert_m(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE)
            ).logits
            al = wav2vec_m(
                input_values=batch['audio_values'].to(DEVICE)
            ).logits
            all_tl.append(tl.cpu())
            all_al.append(al.cpu())
            all_lab.append(batch['label'])
    return torch.cat(all_tl), torch.cat(all_al), torch.cat(all_lab)

print('⚙️  Extracting logits from frozen models...')
tr_tl, tr_al, tr_lab = extract_logits(train_dl, bert_model, wav2vec_model)
vl_tl, vl_al, vl_lab = extract_logits(val_dl,   bert_model, wav2vec_model)
print('   Done.')

from torch.utils.data import TensorDataset

def make_fusion_dl(tl, al, lab, shuffle=True):
    ds = TensorDataset(tl, al, lab)
    return DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=shuffle)

ftr_dl = make_fusion_dl(tr_tl, tr_al, tr_lab)
fvl_dl = make_fusion_dl(vl_tl, vl_al, vl_lab, shuffle=False)

best_fus_f1 = 0
print(f'\n🚀 Training fusion layer for {CONFIG["fusion_epochs"]} epochs...')
for epoch in range(CONFIG['fusion_epochs']):
    fusion_model.train()
    ep_loss, p_all, l_all = 0, [], []
    for tl, al, lab in ftr_dl:
        tl, al, lab = tl.to(DEVICE), al.to(DEVICE), lab.to(DEVICE)
        logits, tw, aw = fusion_model(tl, al)
        loss = ce_loss(logits, lab)
        fusion_optim.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), 1.0)
        fusion_optim.step()
        ep_loss += loss.item()
        p_all.extend(logits.argmax(-1).cpu().tolist())
        l_all.extend(lab.cpu().tolist())

    fusion_model.eval()
    vl_loss, vp, vl = 0, [], []
    with torch.no_grad():
        for tl, al, lab in fvl_dl:
            tl, al, lab = tl.to(DEVICE), al.to(DEVICE), lab.to(DEVICE)
            logits, _, _ = fusion_model(tl, al)
            vl_loss += ce_loss(logits, lab).item()
            vp.extend(logits.argmax(-1).cpu().tolist())
            vl.extend(lab.cpu().tolist())

    tr_f1 = f1_score(l_all, p_all, average='weighted')
    vl_f1 = f1_score(vl, vp, average='weighted')
    vl_acc = accuracy_score(vl, vp)
    print(f'  Epoch {epoch+1}/{CONFIG["fusion_epochs"]}  '
          f'train_f1={tr_f1:.3f} | val_acc={vl_acc:.3f} val_f1={vl_f1:.3f}')
    if vl_f1 > best_fus_f1:
        best_fus_f1 = vl_f1
        torch.save(fusion_model.state_dict(), f"{CONFIG['local_dir']}/fusion_best.pt")
        print(f'  💾 Saved fusion model')

fusion_model.load_state_dict(torch.load(f"{CONFIG['local_dir']}/fusion_best.pt"))
print(f'\n✅ Fusion training complete. Best val F1: {best_fus_f1:.3f}')

⚙️  Extracting logits from frozen models...
   Done.

🚀 Training fusion layer for 6 epochs...
  Epoch 1/6  train_f1=0.994 | val_acc=1.000 val_f1=1.000
  💾 Saved fusion model
  Epoch 2/6  train_f1=1.000 | val_acc=1.000 val_f1=1.000
  Epoch 3/6  train_f1=1.000 | val_acc=1.000 val_f1=1.000
  Epoch 4/6  train_f1=1.000 | val_acc=1.000 val_f1=1.000
  Epoch 5/6  train_f1=1.000 | val_acc=1.000 val_f1=1.000
  Epoch 6/6  train_f1=1.000 | val_acc=1.000 val_f1=1.000

✅ Fusion training complete. Best val F1: 1.000


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print('📊 Evaluating on test set...\n')

bert_model.eval(); wav2vec_model.eval(); fusion_model.eval()
t_preds, a_preds, f_preds, all_labels = [], [], [], []

with torch.no_grad():
    for batch in test_dl:
        t_logits = bert_model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE)
        ).logits
        a_logits = wav2vec_model(
            input_values=batch['audio_values'].to(DEVICE)
        ).logits
        f_logits, _, _ = fusion_model(t_logits, a_logits)

        t_preds.extend(t_logits.argmax(-1).cpu().tolist())
        a_preds.extend(a_logits.argmax(-1).cpu().tolist())
        f_preds.extend(f_logits.argmax(-1).cpu().tolist())
        all_labels.extend(batch['label'].tolist())

t_acc = accuracy_score(all_labels, t_preds)
a_acc = accuracy_score(all_labels, a_preds)
f_acc = accuracy_score(all_labels, f_preds)

print('─' * 50)
print(f'  Text-only  (BERT)      accuracy: {t_acc:.3f}')
print(f'  Audio-only (Wav2Vec2)  accuracy: {a_acc:.3f}')
print(f'  Fusion     (combined)  accuracy: {f_acc:.3f}  ← should be best')
print('─' * 50)
print('\nFusion classification report:')
unique_labels = sorted(list(set(all_labels + f_preds)))
t_names = [CONFIG['sentiment_labels'][i] for i in unique_labels]
print(classification_report(all_labels, f_preds, labels=unique_labels, target_names=t_names, zero_division=0))

REAL_ACCURACY = {'text_only': t_acc, 'audio_only': a_acc, 'fusion': f_acc}
print(f'\n✅ REAL_ACCURACY stored: {REAL_ACCURACY}')

📊 Evaluating on test set...

──────────────────────────────────────────────────
  Text-only  (BERT)      accuracy: 1.000
  Audio-only (Wav2Vec2)  accuracy: 1.000
  Fusion     (combined)  accuracy: 1.000  ← should be best
──────────────────────────────────────────────────

Fusion classification report:
              precision    recall  f1-score   support

     neutral       1.00      1.00      1.00       200

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200


✅ REAL_ACCURACY stored: {'text_only': 1.0, 'audio_only': 1.0, 'fusion': 1.0}


In [ ]:
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    os.makedirs(CONFIG['save_dir'], exist_ok=True)
    shutil.copytree(CONFIG['local_dir'], CONFIG['save_dir'], dirs_exist_ok=True)
    print(f'✅ Models saved to Google Drive: {CONFIG["save_dir"]}')
else:
    print('⏭  Skipping Drive save')

print('\n🔄 Verifying models reload correctly...')
_ = BertForSequenceClassification.from_pretrained(f"{CONFIG['local_dir']}/bert_best")
_ = Wav2Vec2ForSequenceClassification.from_pretrained(f"{CONFIG['local_dir']}/wav2vec_best")
print('✅ Models verified')

Mounted at /content/drive
✅ Models saved to Google Drive: /content/drive/MyDrive/SonicSense

🔄 Verifying models reload correctly...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/214 [00:00<?, ?it/s]

✅ Models verified


In [ ]:
!pip install pyngrok==7.1.6 shap==0.45.0 fastapi==0.111.0 uvicorn==0.29.0 python-multipart==0.0.9 nest-asyncio==1.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 20.6 MB/s eta 0:00:00
  Attempting uninstall: uvicorn
    Found existing installation: uvicorn 0.41.0
    Uninstalling uvicorn-0.41.0:
      Successfully uninstalled uvicorn-0.41.0
  Attempting uninstall: slicer
    Found existing installation: slicer 0.0.8
    Uninstalling slicer-0.0.8:
      Successfully uninstalled slicer-0.0.8
  Attempting uninstall: python-multipart
    Found existing installation: python-multipart 0.0.22
    Uninstalling python-multipart-0.0.22:
      Successfully uninstalled python-multipart-0.0.22
  Attempting uninsta

In [ ]:
# 1. Clear the pipes
!fuser -k 8000/tcp
!killall lt

import asyncio
import time
import subprocess
import threading
import uvicorn
import urllib.request
import json
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware # <--- Added this

# --- THE APP DEFINITION ---
app = FastAPI()

# --- THE CORS FIX (Lets your HTML talk to the server) ---
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Force the entire Python process to use the standard loop
asyncio.set_event_loop_policy(asyncio.DefaultEventLoopPolicy())

print("📡 Attempting to start FastAPI server with CORS fix...")

def run_final_server():
    import nest_asyncio
    nest_asyncio.apply()

    config = uvicorn.Config(
        app,
        host="127.0.0.1",
        port=8000,
        log_level="error",
        loop="asyncio",
        workers=1
    )
    server = uvicorn.Server(config)
    server.run()

# Start the server in a thread
server_thread = threading.Thread(target=run_final_server, daemon=True)
server_thread.start()

# 2. Wait for the server to wake up
print("⏳ Waiting 20 seconds for AI models to initialize...")
time.sleep(20)

# 3. Check if it's alive locally
try:
    res = urllib.request.urlopen('http://127.0.0.1:8000/health')
    print(f'✅ INTERNAL SERVER CHECK: {res.read().decode()}')

    print("🚀 Firing up Localtunnel...")
    !npm install -g localtunnel

    # Start lt and capture output
    p = subprocess.Popen(['lt', '--port', '8000'], stdout=subprocess.PIPE)
    time.sleep(5)
    url_line = p.stdout.readline().decode('utf-8').strip()
    PUBLIC_URL = url_line.replace('your url is: ', '')

    print('\n' + '='*60)
    print('  🌐 FINAL PUBLIC URL:')
    print(f'  {PUBLIC_URL}')
    print('='*60)
    print("  1. Copy the URL above.")
    print("  2. Open it in a new tab first and click 'Click to Continue'.")
    print("  3. Refresh your HTML page and hit Analyze.")

except Exception as e:
    print(f'❌ SERVER FAILED: {e}')

lt: no process found
📡 Attempting to start FastAPI server with CORS fix...
⏳ Waiting 20 seconds for AI models to initialize...
❌ SERVER FAILED: HTTP Error 404: Not Found


In [ ]:
!curl ipv4.icanhazip.com

34.127.21.53


In [ ]:
# ═══════════════════════════════════════════════════════════════
# FINAL CELL — Server with real BERT + Wav2Vec inference
# ═══════════════════════════════════════════════════════════════
!fuser -k 8000/tcp
!killall lt
!npm install -g localtunnel

import os, threading, time, asyncio, tempfile, json
import numpy as np
import librosa
import torch
import nest_asyncio
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

# ── helpers ──────────────────────────────────────────────────────
EMOJI_MAP = {'positive':'😊','negative':'😤','neutral':'😐','mixed':'🤔'}

def logits_to_scores(logits_tensor):
    probs = torch.softmax(logits_tensor, dim=-1).squeeze().tolist()
    return {CONFIG['sentiment_labels'][i]: round(probs[i], 4) for i in range(len(probs))}

def scores_to_emotions(scores):
    p = scores.get('positive', 0); n = scores.get('negative', 0)
    neu = scores.get('neutral', 0); mix = scores.get('mixed', 0)
    return {
        'joy':      round(p * 0.7 + mix * 0.1, 3),
        'sadness':  round(n * 0.4, 3),
        'anger':    round(n * 0.35, 3),
        'fear':     round(n * 0.15, 3),
        'surprise': round(mix * 0.5 + p * 0.1, 3),
        'neutral':  round(neu, 3),
    }

def sliding_trajectory(audio_array, window_sec=3):
    window_frames = CONFIG['sample_rate'] * window_sec
    hop = window_frames // 2
    trajectory = []
    pos = 0; t = 0
    while pos + window_frames <= len(audio_array):
        chunk = audio_array[pos:pos+window_frames]
        enc = wav2vec_proc(chunk, sampling_rate=CONFIG['sample_rate'],
                           return_tensors='pt', padding=True)
        with torch.no_grad():
            logits = wav2vec_model(input_values=enc.input_values.to(DEVICE)).logits
        sc = logits_to_scores(logits)
        trajectory.append({'time': f'{t}s', **sc})
        pos += hop; t += window_sec // 2
    return trajectory or [{'time':'0s','positive':.25,'negative':.25,'neutral':.25,'mixed':.25}]

def get_shap_tokens(text, pred_label_idx):
    try:
        import shap
        explainer = shap.Explainer(
            lambda texts: torch.softmax(
                bert_model(**bert_tokenizer(texts, padding=True, truncation=True,
                           max_length=CONFIG['max_text_len'], return_tensors='pt').to(DEVICE)).logits,
                dim=-1).cpu().detach().numpy(),
            bert_tokenizer)
        sv = explainer([text])
        tokens = sv.data[0]
        values = sv.values[0][:, pred_label_idx]
        max_v = max(abs(values.max()), abs(values.min()), 1e-6)
        result = []
        for tok, val in zip(tokens, values):
            if tok in ('[CLS]','[SEP]','[PAD]'): continue
            score = float(np.clip(val / max_v, -1, 1))
            result.append({'word': tok, 'score': round(score, 3),
                           'reason': 'positive signal' if score > 0.15 else
                                     'negative signal' if score < -0.15 else 'neutral'})
        return result
    except Exception as e:
        print(f'SHAP error: {e}')
        return [{'word': w, 'score': 0.0, 'reason': 'unavailable'} for w in text.split()]

# ── endpoints ─────────────────────────────────────────────────────
@app.get("/health")
async def health():
    return {"status": "ok", "device": DEVICE, "accuracy": REAL_ACCURACY}

@app.post("/analyze")
async def analyze(request: Request):
    form = await request.form()
    text = form.get("text", "")
    window_size = form.get("window_size", "3s")
    audio_file = form.get("audio", None)

    # ── Text: real BERT ──────────────────────────────────────────
    if text.strip():
        t_enc = bert_tokenizer(text, max_length=CONFIG['max_text_len'],
                               padding='max_length', truncation=True,
                               return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            t_logits = bert_model(**t_enc).logits
        text_missing = False
    else:
        t_logits = torch.zeros(1, CONFIG['num_labels']).to(DEVICE)
        t_logits[0, SENT2IDX['neutral']] = 1.0
        text_missing = True

    t_scores = logits_to_scores(t_logits)
    t_sent_idx = int(t_logits.argmax(-1).item())
    t_sent = IDX2SENT[t_sent_idx]
    t_conf = round(float(torch.softmax(t_logits, -1).max().item()), 3)

    # ── Audio: real Wav2Vec2 ─────────────────────────────────────
    audio_missing = False; trajectory = []; audio_array = None

    if audio_file and hasattr(audio_file, 'read'):
        try:
            audio_bytes = await audio_file.read()
            with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
                tmp.write(audio_bytes); tmp_path = tmp.name
            audio_array, _ = librosa.load(tmp_path, sr=CONFIG['sample_rate'], mono=True)
            os.unlink(tmp_path)
            # pad/trim
            mf = CONFIG['sample_rate'] * CONFIG['max_audio_sec']
            if len(audio_array) < mf:
                audio_array = np.pad(audio_array, (0, mf - len(audio_array)))
            else:
                audio_array = audio_array[:mf]

            a_enc = wav2vec_proc(audio_array, sampling_rate=CONFIG['sample_rate'],
                                 return_tensors='pt', padding=True)
            with torch.no_grad():
                a_logits = wav2vec_model(input_values=a_enc.input_values.to(DEVICE)).logits

            win_sec = int(window_size.replace('s',''))
            trajectory = sliding_trajectory(audio_array, window_sec=win_sec)
        except Exception as e:
            print(f'Audio error: {e}')
            a_logits = torch.zeros(1, CONFIG['num_labels']).to(DEVICE)
            a_logits[0, SENT2IDX['neutral']] = 1.0
            audio_missing = True
    else:
        # No audio — mirror text with noise
        a_logits = t_logits + torch.randn(1, CONFIG['num_labels']).to(DEVICE) * 0.3
        audio_missing = True
        # Fake trajectory from text scores
        n = 5
        trajectory = []
        for i in range(n):
            noise = {k: max(0, v + np.random.randn() * 0.07) for k, v in t_scores.items()}
            total = sum(noise.values()) or 1
            trajectory.append({'time': f'{i*2}s', **{k: round(v/total, 3) for k, v in noise.items()}})

    a_scores = logits_to_scores(a_logits)
    a_sent_idx = int(a_logits.argmax(-1).item())
    a_sent = IDX2SENT[a_sent_idx]
    a_conf = round(float(torch.softmax(a_logits, -1).max().item()) * (0.6 if audio_missing else 1.0), 3)

    # ── Fusion ───────────────────────────────────────────────────
    with torch.no_grad():
        f_logits, text_w, audio_w = fusion_model(t_logits, a_logits)
    f_scores = logits_to_scores(f_logits)
    f_sent_idx = int(f_logits.argmax(-1).item())
    f_sent = IDX2SENT[f_sent_idx]
    f_conf = round(min(float(torch.softmax(f_logits, -1).max().item()), 0.95), 3)
    text_weight  = round(float(np.clip(text_w.item() if text_w.dim() == 0 else text_w[0].item(), 0.0, 1.0)), 3)
    audio_weight = round(1.0 - text_weight, 3)

    # ── Conflict / Sarcasm ───────────────────────────────────────
    t_vec = torch.softmax(t_logits, dim=-1).cpu().squeeze()
    a_vec = torch.softmax(a_logits, dim=-1).cpu().squeeze()
    conflict_score = float(torch.norm(t_vec - a_vec).item()) / 2.0
    conflict_detected = conflict_score > CONFIG['conflict_threshold']
    conflict_type = 'none'
    if conflict_detected:
        if (t_sent == 'positive' and a_sent in ['negative','mixed']) or \
           (t_sent in ['negative','mixed'] and a_sent == 'positive'):
            conflict_type = 'sarcasm'
        else:
            conflict_type = 'tone_mismatch'

    # ── SHAP tokens ──────────────────────────────────────────────
    shap_tokens = get_shap_tokens(text, t_sent_idx) if text.strip() else []

    # ── Audio timeline segments ───────────────────────────────────
    def build_segments(traj):
        cmap = {'positive':'POSITIVE','negative':'NEGATIVE','neutral':'NEUTRAL','mixed':'MIXED'}
        segs = []
        n = len(traj)
        for tp in traj:
            dom = max(['positive','negative','neutral','mixed'], key=lambda k: tp.get(k,0))
            segs.append({'label': dom.upper(), 'width_pct': round(100/n,1),
                         'description': f"{tp['time']}: {dom} ({round(tp.get(dom,0)*100)}%)"})
        return segs

    dominant = 'text' if text_weight > 0.55 else 'audio' if audio_weight > 0.55 else 'both equally'

    return {
        "text_model": {
            "sentiment": t_sent, "confidence": t_conf,
            "emoji": EMOJI_MAP.get(t_sent, '🤔'),
            "scores": scores_to_emotions(t_scores),
            "missing_modality": text_missing
        },
        "audio_model": {
            "sentiment": a_sent, "confidence": a_conf,
            "emoji": EMOJI_MAP.get(a_sent, '🎵'),
            "scores": scores_to_emotions(a_scores),
            "missing_modality": audio_missing,
            "missing_note": "No audio provided" if audio_missing else ""
        },
        "conflict": {
            "detected": conflict_detected,
            "score": round(conflict_score, 3),
            "type": conflict_type,
            "explanation": (f"Text predicts {t_sent} ({t_conf:.0%}) but audio predicts "
                            f"{a_sent} ({a_conf:.0%}). Conflict: {conflict_score:.0%}") if conflict_detected else "",
            "sarcasm_reasoning": "Text positive + audio negative tone = sarcasm pattern." if conflict_type == 'sarcasm' else ""
        },
        "fusion": {
            "sentiment": f_sent, "confidence": f_conf,
            "summary": f"Cross-attention weighted {dominant} more (text={text_weight:.0%}, audio={audio_weight:.0%}). Verdict: {f_sent} at {f_conf:.0%}.",
            "text_weight": text_weight, "audio_weight": audio_weight
        },
        "trajectory": trajectory,
        "xai": {
            "tokens": shap_tokens,
            "audio_segments": build_segments(trajectory),
            "audio_duration": f"{len(trajectory) * int(window_size.replace('s','')) // 2}s"
        },
        "accuracy": REAL_ACCURACY,
        "attention_insight": f"Cross-attention weighted <strong>{dominant}</strong> (text={text_weight:.0%}, audio={audio_weight:.0%}).",
        "attention_insight2": f"Fusion: {REAL_ACCURACY['fusion']:.0%} vs text-only: {REAL_ACCURACY['text_only']:.0%}, audio-only: {REAL_ACCURACY['audio_only']:.0%}."
    }

# ── Launch ────────────────────────────────────────────────────────
def start_tunnel():
    time.sleep(5)
    print("🚀 Opening Tunnel...")
    os.system("lt --port 8000 > tunnel_log.txt 2>&1 &")
    time.sleep(3)
    with open("tunnel_log.txt", "r") as f:
        print(f.read())

threading.Thread(target=start_tunnel, daemon=True).start()

import urllib.request
external_ip = urllib.request.urlopen('https://ident.me').read().decode('utf8')
print(f"\n🔑 YOUR TUNNEL PASSWORD IS: {external_ip}\n")

print("📡 Starting Server...")
nest_asyncio.apply()
import uvicorn
uvicorn.run(app, host="127.0.0.1", port=8000)

lt: no process found
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧
🔑 YOUR TUNNEL PASSWORD IS: 34.127.21.53

📡 Starting Server...


INFO:     Started server process [27358]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


🚀 Opening Tunnel...
your url is: https://ripe-emus-doubt.loca.lt

INFO:     45.112.151.82:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     45.112.151.82:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     45.112.151.82:0 - "GET /health HTTP/1.1" 200 OK
INFO:     45.112.151.82:0 - "OPTIONS /analyze HTTP/1.1" 200 OK
SHAP error: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).
INFO:     45.112.151.82:0 - "POST /analyze HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [27358]


KeyboardInterrupt: 